# 1. Connect to a postgreSQL database using *psycopg* with login data stored in an environment variable.

In [1]:
import os
import psycopg2

DATABASE_URL = os.getenv("DATABASE_URL")

conn = psycopg2.connect(DATABASE_URL)

print(conn)

<connection object at 0x00000277B6A93780; dsn: 'user=postgres password=xxx dbname=exercises host=localhost port=5432', closed: 0>


# 2. Create a table called *workers* with fields *name*, *surname*, *position*, *hire date* and *salary*. Check if the table already exists. Insert a constraint for rows to be unique with name and surname. 

In [2]:

DATABASE_URL = os.getenv("DATABASE_URL")

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

create_table_sql = """
CREATE TABLE IF NOT EXISTS workers (
    worker_id BIGSERIAL PRIMARY KEY,
    name VARCHAR(25),
    surname VARCHAR(50),
    position VARCHAR(150),
    hire_date DATE,
    salary NUMERIC,
    CONSTRAINT uq_worker_name UNIQUE (name, surname)
);
"""

cur.execute(create_table_sql)
conn.commit()

cur.execute("SELECT * FROM workers;")

rows = cur.fetchall()

print("connected")
print(len(rows))

for row in rows:
    print(row)

connected
0


# 3. Add 5 rows to the tabe *workers*.

In [3]:
data = [
    ('Jaden', 'Clark', 'Software Engineer', '2012-09-30', 65000),
    ('Magnus', 'Kampinsky', 'Software Architect', '1996-07-21', 82000),
    ('Olivia', 'Turner', 'Data Scientist', '2015-03-14', 71000),
    ('Ethan', 'Nguyen', 'Backend Developer', '2018-11-02', 68000),
    ('Sophia', 'Martinez', 'DevOps Engineer', '2013-06-25', 75000)
]

insert_sql = """
INSERT INTO workers (name, surname, position, hire_date, salary)
VALUES (%s, %s, %s, %s, %s)
ON CONFLICT (name, surname) DO NOTHING;
"""

cur.executemany(insert_sql, data)
conn.commit()

cur.execute("SELECT * FROM workers;")
rows = cur.fetchall()

for row in rows:
    print(row)


(1, 'Jaden', 'Clark', 'Software Engineer', datetime.date(2012, 9, 30), Decimal('65000'))
(2, 'Magnus', 'Kampinsky', 'Software Architect', datetime.date(1996, 7, 21), Decimal('82000'))
(3, 'Olivia', 'Turner', 'Data Scientist', datetime.date(2015, 3, 14), Decimal('71000'))
(4, 'Ethan', 'Nguyen', 'Backend Developer', datetime.date(2018, 11, 2), Decimal('68000'))
(5, 'Sophia', 'Martinez', 'DevOps Engineer', datetime.date(2013, 6, 25), Decimal('75000'))


# 4. Create another table called projects_assignment, consisting of columns *project_id*, *project_name*, *assignee_ids* and *assignment_time_ratio*. Fill it with 3 rows with distinct projects.

In [12]:
create_table_sql = """
CREATE TABLE IF NOT EXISTS projects_assignment (
    project_id BIGSERIAL PRIMARY KEY,
    project_name VARCHAR(150),
    assignee_ids BIGINT[],
    assignment_time_ratio NUMERIC[],
    CONSTRAINT uq_project_name UNIQUE (project_name)
);
"""

cur.execute(create_table_sql)
conn.commit()

data = [
    ('Platform Front-End', [1,2,5], [100.0,50.0,30.0]),
    ('Platform Back-End', [4,2,5], [100.0,50.0,30.0]),
    ('Platform ML Analytics', [3,5], [100.0,40.0])
]

insert_sql = """
INSERT INTO projects_assignment (project_name, assignee_ids, assignment_time_ratio)
VALUES (%s, %s, %s)
ON CONFLICT (project_name) DO NOTHING;
"""

cur.executemany(insert_sql, data)
conn.commit()

cur.execute("SELECT * FROM projects_assignment;")

rows = cur.fetchall()

print("connected")
print(len(rows))

for row in rows:
    print(row)

connected
3
(1, 'Platform Front-End', [1, 2, 5], [Decimal('100.0'), Decimal('50.0'), Decimal('30.0')])
(2, 'Platform Back-End', [4, 2, 5], [Decimal('100.0'), Decimal('50.0'), Decimal('30.0')])
(3, 'Platform ML Analytics', [3, 5], [Decimal('100.0'), Decimal('40.0')])


# 5. Retrieve the columns with surnames and positions lexicographically sorted by surnames from the *workers* table.

In [16]:
cur.execute("SELECT surname, position FROM workers ORDER BY surname;")
rows = cur.fetchall()

for row in rows:
    print(row)

('Clark', 'Software Engineer')
('Kampinsky', 'Software Architect')
('Martinez', 'DevOps Engineer')
('Nguyen', 'Backend Developer')
('Turner', 'Data Scientist')


# 6. Retrieve the workers names and surnames with salaries smaller or equal 70000.

In [20]:
cur.execute("SELECT name, surname, salary FROM workers WHERE salary <= 70000  ORDER BY salary;")
rows = cur.fetchall()

for row in rows:
    print(row)

('Jaden', 'Clark', Decimal('65000'))
('Ethan', 'Nguyen', Decimal('68000'))


# 7. Get the list of workers that do not have the word 'software' in their position description.

In [22]:
cur.execute("SELECT name, surname, position FROM workers WHERE position NOT ILIKE '%software%';")
rows = cur.fetchall()

for row in rows:
    print(row)

('Olivia', 'Turner', 'Data Scientist')
('Ethan', 'Nguyen', 'Backend Developer')
('Sophia', 'Martinez', 'DevOps Engineer')
